# Chatbot 

In [ ]:
!pip install -U agno openai

In [ ]:
import os
os.environ["NVIDIA_API_KEY"] = ""  # <-- paste your key

---
## 1. Helper: Create Agent with a Given Temperature

In [ ]:
from agno.agent import Agent
from agno.models.nvidia import Nvidia

def make_agent(temperature: float = 0.7, system_prompt: str = "You are a helpful assistant.") -> Agent:
    return Agent(
        model=Nvidia(
            id="minimaxai/minimax-m3",
            options={"temperature": temperature},
        ),
        instructions=[system_prompt],
        markdown=True,
    )

---
## 2. Temperature Comparison

Same prompt, different temperatures.

In [ ]:
prompt = "Explain what a neural network is in 2-3 sentences."

for temp in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n{'='*60}")
    print(f"  TEMPERATURE = {temp}")
    print(f"{'='*60}")
    agent = make_agent(temperature=temp)
    agent.print_response(prompt)

---
## 3. Prompt Techniques

### 3a. Zero-Shot Prompting

In [ ]:
agent = make_agent(temperature=0.3)

zero_shot_prompt = """Classify the following review as POSITIVE, NEGATIVE, or NEUTRAL:
"The product arrived on time but the quality was mediocre."

Respond with just the label."""

agent.print_response(zero_shot_prompt)

### 3b. Few-Shot Prompting

In [ ]:
few_shot_prompt = """Classify reviews as POSITIVE, NEGATIVE, or NEUTRAL.

Examples:
- "Absolutely love this! Best purchase ever." -> POSITIVE
- "Terrible experience, never buying again." -> NEGATIVE
- "It works fine, nothing special." -> NEUTRAL

Now classify:
- "The product arrived on time but the quality was mediocre." ->"""

agent.print_response(few_shot_prompt)

### 3c. Chain-of-Thought (CoT) Prompting

In [ ]:
cot_agent = make_agent(temperature=0.2)

cot_prompt = """Solve this step by step:

A store sells notebooks for $4 each. If you buy 3 or more, you get 20% off.
How much does it cost to buy 5 notebooks?

Think through each step before giving the final answer."""

cot_agent.print_response(cot_prompt)

### 3d. Role-Playing Prompt

In [ ]:
role_agent = make_agent(
    temperature=0.5,
    system_prompt="You are a senior Python developer who gives concise, practical advice."
)

role_prompt = "How should I handle exceptions in a production API?"

role_agent.print_response(role_prompt)

### 3e. Structured Output Prompt

In [ ]:
structured_agent = make_agent(temperature=0.1)

structured_prompt = """Return a JSON object with exactly these keys for the person below:
- name
- age
- occupation

Person: John Smith, a 35-year-old software engineer.

Return ONLY the JSON, no explanation."""

structured_agent.print_response(structured_prompt)

### 3f. Self-Consistency (Run Same Prompt Multiple Times)

In [ ]:
agent_consistency = make_agent(temperature=0.9)

consistency_prompt = """Answer with just the number:
What is 15 + 27?"""

print("Running 3 times at temp=0.9:")
for i in range(3):
    print(f"\n--- Run {i+1} ---")
    agent_consistency.print_response(consistency_prompt)

---
## 4. Creative vs Precise — Temperature Sweep

In [ ]:
creative_prompt = "Write a one-sentence story about a robot."

for temp in [0.0, 0.7, 1.2]:
    print(f"\n{'='*60}")
    print(f"  TEMP = {temp}  ({'precise' if temp < 0.5 else 'balanced' if temp < 1.0 else 'creative'})")
    print(f"{'='*60}")
    a = make_agent(temperature=temp)
    a.print_response(creative_prompt)

---
## 5. Interactive Chat

Run the cell below to chat interactively. Type `quit` to exit.

In [ ]:
chat_agent = make_agent(temperature=0.7)

print("Interactive chatbot (type 'quit' to exit):")
print("-" * 40)

while True:
    user_input = input("You: ")
    if user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    chat_agent.print_response(user_input)